In [0]:
CREATE OR REPLACE VIEW gold.analistas.vw_clisp_SLA_producao AS
WITH 

-- =====================================================================================
-- METADADOS DA OP
-- =====================================================================================
DADOS_OP AS (
    SELECT 
        op.OrdemProducao,
        op.StatusOrdemProducao,
        op.CodProcessoUnico,
        op.NumUnicoNotaPedido, 
        op.DataHoraInclusao AS DataInclusao_OP,
        COALESCE(op.DataHoraInicializacao, op.DataHoraInclusao) AS DataInicio_OP,
        op.DataHoraFinalizacao AS DataFim_OP,
        COALESCE(op.DataHoraInicializacao, op.DataHoraInclusao) + INTERVAL 5 DAYS AS Prazo_SLA_Calculado
    FROM gold.sankhya.fato_ordem_producao op
    WHERE op.DataHoraInclusao >= '2025-01-01'
),

-- =====================================================================================
-- EXPEDIÇÃO LOGÍSTICA
-- =====================================================================================
EXPEDICAO_MAX AS (
    SELECT
        NumUnicoNota,
        MAX(DataExpedicao) AS DataExpedicao
    FROM gold.sankhya.fato_itens_notas_expedidas
    GROUP BY NumUnicoNota
),

-- =====================================================================================
-- METADADOS DE SLA PARA OPs LEGADAS DE 2025 (fato_ordem_producao só cobre a partir de ~2025-12-30)
-- Usa a data de inclusão da série como proxy: 100% de cobertura e cronologicamente coerente
-- (anterior ao apontamento em 99,99% dos casos)
-- =====================================================================================
DADOS_OP_LEGADO_2025 AS (
    SELECT
        OrdemProducao,
        MIN(DataHoraInclusao) AS Data_Inclusao_Legado,
        MIN(DataHoraInclusao) AS Data_Inicio_Legado,
        MIN(DataHoraInclusao) + INTERVAL 5 DAYS AS Prazo_SLA_Legado
    FROM gold.sankhya.fato_ordem_producao_seriepa_ciclo
    GROUP BY OrdemProducao
)

-- =====================================================================================
-- CONSOLIDAÇÃO FINAL: dados de produção espelhados diretamente das views
-- gold.analistas.vw_producaocontagem2025 e vw_producaocontagem2026
-- (já contêm CodProd, atributos de produto, Qtd_Produzida/Planejada/Saldo corretos)
-- =====================================================================================
SELECT *
FROM (

    -- Espelha vw_producaocontagem2025
    SELECT
        '2025' AS Origem_Base,
        V.OP,
        V.CodProd AS CodProduto,
        V.DescricaoProduto AS Descricao,
        V.Familia,
        V.LinhaDeNegocio AS Linha,
        V.Ano,
        V.Mes,
        V.Semana,
        V.Qtd_Produzida,
        V.Qtd_Planejada,
        V.Saldo,

        COALESCE(OP.DataInclusao_OP, LEG.Data_Inclusao_Legado) AS Data_Inclusao,
        COALESCE(OP.DataInicio_OP, LEG.Data_Inicio_Legado) AS Data_Inicio,
        COALESCE(OP.Prazo_SLA_Calculado, LEG.Prazo_SLA_Legado) AS Limite_SLA,
        V.DataProducao AS Data_Producao,
        V.DataHoraApontamento AS DataHora_Apontamento,

        -- 🚀 Solução Estável: FROM_UNIXTIME calcula os segundos e converte direto para String HH:mm:ss
        FROM_UNIXTIME(
            COALESCE(UNIX_TIMESTAMP(V.DataHoraApontamento) - UNIX_TIMESTAMP(COALESCE(OP.DataInicio_OP, LEG.Data_Inicio_Legado)), 0),
            'HH:mm:ss'
        ) AS LT_Minutos,
        COALESCE(DATEDIFF(V.DataProducao, CAST(COALESCE(OP.DataInicio_OP, LEG.Data_Inicio_Legado) AS DATE)), 0) AS LT_Dias,

        CASE 
            WHEN COALESCE(OP.DataInicio_OP, LEG.Data_Inicio_Legado) IS NULL THEN 'OP NÃO INICIALIZADA'
            WHEN V.DataProducao <= CAST(COALESCE(OP.Prazo_SLA_Calculado, LEG.Prazo_SLA_Legado) AS DATE) THEN 'DENTRO DO SLA'
            ELSE 'FORA DO SLA'
        END AS Status_SLA,

        V.NumUnicoNota,
        V.NumeroNota,
        V.DataFaturamento,
        CAST(EXP.DataExpedicao AS DATE) AS DataExpedicao

    FROM gold.analistas.vw_producaocontagem2025 V
    LEFT JOIN DADOS_OP OP ON V.OP = OP.OrdemProducao
    LEFT JOIN DADOS_OP_LEGADO_2025 LEG ON V.OP = LEG.OrdemProducao
    LEFT JOIN EXPEDICAO_MAX EXP ON V.NumUnicoNota = EXP.NumUnicoNota

    UNION ALL

    -- Espelha vw_producaocontagem2026
    SELECT
        '2026' AS Origem_Base,
        V.OP,
        V.CodProd AS CodProduto,
        V.DescricaoProduto AS Descricao,
        V.Familia,
        V.LinhaDeNegocio AS Linha,
        V.Ano,
        V.Mes,
        V.Semana,
        V.Qtd_Produzida,
        V.Qtd_Planejada,
        V.Saldo,

        OP.DataInclusao_OP AS Data_Inclusao,
        OP.DataInicio_OP AS Data_Inicio,
        OP.Prazo_SLA_Calculado AS Limite_SLA,
        V.DataProducao AS Data_Producao,
        V.DataHoraApontamento AS DataHora_Apontamento,

        FROM_UNIXTIME(
            COALESCE(UNIX_TIMESTAMP(V.DataHoraApontamento) - UNIX_TIMESTAMP(OP.DataInicio_OP), 0),
            'HH:mm:ss'
        ) AS LT_Minutos,
        COALESCE(DATEDIFF(V.DataProducao, CAST(OP.DataInicio_OP AS DATE)), 0) AS LT_Dias,

        CASE 
            WHEN OP.DataInicio_OP IS NULL THEN 'OP NÃO INICIALIZADA'
            WHEN V.DataProducao <= CAST(OP.Prazo_SLA_Calculado AS DATE) THEN 'DENTRO DO SLA'
            ELSE 'FORA DO SLA'
        END AS Status_SLA,

        V.NumUnicoNota,
        V.NumeroNota,
        V.DataFaturamento,
        CAST(EXP.DataExpedicao AS DATE) AS DataExpedicao

    FROM gold.analistas.vw_producaocontagem2026 V
    LEFT JOIN DADOS_OP OP ON V.OP = OP.OrdemProducao
    LEFT JOIN EXPEDICAO_MAX EXP ON V.NumUnicoNota = EXP.NumUnicoNota

) X
ORDER BY 
    DataHora_Apontamento DESC, 
    OP, 
    CodProduto;